In [1]:
import torch
import utils
from pathlib import Path
import preprocess_data as ppd
from process_session import session as ss
import process_probe as pp
import process_attribution as pa
import numpy as np
import plots
import importlib
import matplotlib.pyplot as plt
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import pandas as pd
output_dir = "F:/vbn_s3_cache/"
df = pd.read_csv('units_info.csv')
ripple_times = np.load(r"F:\ripple_times.npy")

In [2]:
session_id = 1044385384
session_obj = ss(session_id, df, output_dir)

In [3]:
input_size = len(session_obj.units)
hidden_size = 64 # 
num_layers = 2
seqlength = 750
num_epochs = 15 # 
bands = [(0.5, 4), (4, 8), (8, 12), (12, 25), (25, 50), (50, 100), (100, 200), (200, 400)]
batch_size = 8

In [4]:
bin_size=0.0008
spikes_obj = ppd.pre_process_spikes(session_obj.units, session_obj.spike_times, bin_size=bin_size, sigma=3)
spikes_obj.getSpkMat(session_obj.passive_times[0], session_obj.passive_times[1])
spikes_obj.truncate(seqlength)
spikes_obj.convolve_with_gaussian()
spikes_obj.zscore()

100%|██████████| 623/623 [03:22<00:00,  3.07it/s]


In [15]:
lfp_obj = ppd.pre_process_lfp(session_id, session_obj.channels, session_obj.passive_times[0], session_obj.passive_times[1], output_dir)
lfp_obj.filter_lfp()
lfp_obj.downsample_lfp(5)
lfp_obj.align_lfp(spikes_obj.spkMat.shape[0])

100%|██████████| 8/8 [02:53<00:00, 21.65s/it]


In [5]:
import numpy as np
from typing import List, Tuple

def find_time_points(start: float, stop: float, given_times: List[float], boundary: float, n: int) -> List[float]:
    """
    Finds up to n candidate time points x in [start, stop] such that 
    the window [x-boundary, x+boundary] does not touch any given time point's window
    [t-boundary, t+boundary]. In other words, for each given time t, we require that
    [x-boundary, x+boundary] and [t-boundary, t+boundary] do not overlap or even touch.
    
    This is achieved by requiring that x is not in any interval [t - 2*boundary, t + 2*boundary].
    Also, x must be chosen so that its own window lies within [start, stop].
    
    Parameters:
      start (float): the start time.
      stop (float): the stop time.
      given_times (List[float]): a list of time points that already exist.
      boundary (float): the boundary length for both candidate and given time points.
      n (int): the desired number of candidate points.
      
    Returns:
      List[float]: a list of candidate time points (of length <= n) satisfying the conditions.
    """
    # The candidate's window must lie completely in [start, stop]:
    overall_start = start + boundary
    overall_stop  = stop - boundary
    if overall_start >= overall_stop:
        return []
    
    # For each given time t, the candidate x is invalid if it lies in [t - 2*boundary, t + 2*boundary]
    forbidden_intervals: List[Tuple[float, float]] = []
    for t in given_times:
        forbidden_intervals.append((t - 2*boundary, t + 2*boundary))
    
    # Sort intervals by their starting point
    forbidden_intervals.sort(key=lambda interval: interval[0])
    
    # Merge overlapping or touching forbidden intervals
    merged_forbidden: List[Tuple[float, float]] = []
    for interval in forbidden_intervals:
        if not merged_forbidden:
            merged_forbidden.append(interval)
        else:
            last_start, last_end = merged_forbidden[-1]
            if interval[0] <= last_end:  # intervals overlap or touch
                merged_forbidden[-1] = (last_start, max(last_end, interval[1]))
            else:
                merged_forbidden.append(interval)
    
    # Subtract the forbidden intervals (clipped to the overall candidate region)
    safe_intervals: List[Tuple[float, float]] = []
    current = overall_start
    for (fstart, fend) in merged_forbidden:
        # Skip forbidden intervals that lie completely outside our overall region.
        if fend < overall_start or fstart > overall_stop:
            continue
        # Clip the forbidden interval to our overall region
        fstart_clipped = max(fstart, overall_start)
        fend_clipped   = min(fend, overall_stop)
        # The region from current to the start of the forbidden interval is safe.
        if current < fstart_clipped:
            safe_intervals.append((current, fstart_clipped))
        # Update current to the end of the forbidden region
        current = max(current, fend_clipped)
        if current >= overall_stop:
            break
    if current < overall_stop:
        safe_intervals.append((current, overall_stop))
    
    # From safe_intervals, choose candidate time points.
    # Note: any candidate x chosen from a safe interval is valid.
    result: List[float] = []
    for (s, e) in safe_intervals:
        count_needed = n - len(result)
        if count_needed <= 0:
            break
        
        # If only one candidate is needed from this interval, pick the midpoint.
        if count_needed == 1:
            candidate = (s + e) / 2.0
            result.append(candidate)
        else:
            # If more than one candidate is needed, sample evenly within (s, e)
            # We avoid the exact endpoints to ensure the candidate's window does not "touch" the boundaries.
            pts = np.linspace(s, e, count_needed + 2)[1:-1]  # skip the endpoints
            result.extend(pts.tolist())
        if len(result) >= n:
            result = result[:n]
            break
    return result

In [7]:
input_size = len(session_obj.units)
hidden_size = 64 # 
num_layers = 2
seqlength = 400
num_epochs = 15 # 
bands = [(0.5, 4), (4, 8), (8, 12), (12, 25), (25, 50), (50, 100), (100, 200), (200, 400)]
batch_size = 8

In [13]:
# to generate the true label data
# for each ripple, we can first construct training data with the ripple at the center, seqlength = 750
# then, we can shift the window by +/- 10-200, step=5 bins to generate more training data
r = ripple_times[ripple_times > session_obj.passive_times[0]]
r = r[r < session_obj.passive_times[1]]
multiply = 5
step_size = 20
x_train_true = np.zeros((len(r)*multiply*2, seqlength, input_size), dtype=np.float32)
y_train_true = np.ones((len(r)*multiply*2, 1))
for i, ripple in enumerate(r):
    ripple_start_idx = np.searchsorted(spikes_obj.timestamps, ripple-seqlength/2*bin_size)
    ripple_end_idx = ripple_start_idx + seqlength
    for jdx, j in enumerate(range(-multiply, multiply, 1)): # from -100 to 100, step=10
        x_train_true[int(i*multiply+jdx)] = spikes_obj.spkMat[int(ripple_start_idx+j*step_size):int(ripple_end_idx+j*step_size)]

print(x_train_true.shape)
# there are around 300 ripples in the session, and with augmentation, we can generate around 10k training samples for true labels

(3240, 400, 623)


In [14]:
# then, we can find time points with no ripples, and generate the same number of training samples for false labels
# find time points with no ripples but also +/- multiple * step_size bins with no ripples
# we can use the same step_size as before
x_train_false = np.zeros((len(r)*multiply*2, seqlength, input_size), dtype=np.float32)
y_train_false = np.zeros((len(r)*multiply*2, 1))
time_points = find_time_points(session_obj.passive_times[0], session_obj.passive_times[1], r, multiply*step_size*bin_size, len(r))
for i, time_point in enumerate(time_points):
    t_start_idx = np.searchsorted(spikes_obj.timestamps, time_point)
    t_end_idx = t_start_idx + seqlength
    for jdx, j in enumerate(range(-multiply, multiply, 1)): # from -100 to 100, step=10
        x_train_false[int(i*multiply+jdx)] = spikes_obj.spkMat[int(t_start_idx+j*step_size):int(t_end_idx+j*step_size)]

print(x_train_false.shape)

(3240, 400, 623)


In [15]:
from sklearn.model_selection import train_test_split

In [16]:
X_train = np.concatenate((x_train_true, x_train_false), axis=0, dtype=np.float32)
del x_train_true, x_train_false
y_train = np.concatenate((y_train_true, y_train_false), axis=0, dtype=np.float32)
del y_train_true, y_train_false
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [78]:
np.save("X_test.npy", X_test)

In [37]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return self.sigmoid(out)

    def predict(self, X):
        self.eval()
        X = torch.tensor(X, dtype=torch.float32)
        with torch.no_grad():
            y_pred = self.forward(X)
        return np.round(y_pred.cpu().numpy())

def train_model(model, X, y, epochs=10, batch_size=32, device="cpu"):
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)  # Ensure correct shape

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    dataloader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)

    model.train()
    model.to(device)

    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        epoch_loss /= len(dataloader)
        losses.append(epoch_loss)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

    return losses

In [50]:
def evaluate_model(model, X_test, y_test, device="cpu"):
    model.eval()  # Set model to evaluation mode
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)  # Ensure shape (N,1)

    with torch.no_grad():  # Disable gradient computation
        y_pred_probs = model(X_test)  # Probabilities (between 0 and 1)
    
    y_pred_binary = (y_pred_probs >= 0.5).float()  # Convert to 0 or 1

    return y_pred_binary.cpu().numpy(), y_pred_probs.cpu().numpy()

def compute_accuracy(y_true, y_pred):
    accuracy = np.mean(y_true == y_pred)
    return accuracy


In [62]:
input_size = len(session_obj.units)
hidden_size = 256 # 
num_layers = 3
seqlength = 750
num_epochs = 15 # 
bands = [(0.5, 4), (4, 8), (8, 12), (12, 25), (25, 50), (50, 100), (100, 200), (200, 400)]
epochs = 15       # Number of training epochs
batch_size = 4   # Batch size

In [63]:
model = LSTM(input_size, hidden_size, num_layers).to(device)

In [64]:
losses = train_model(model, X_train, y_train, epochs=epochs, batch_size=batch_size, device=device)

Epoch 1/15, Loss: 0.4479


KeyboardInterrupt: 

In [59]:
# Get predictions
y_pred_binary, _ = evaluate_model(model, X_test, y_test, device)

# Compute accuracy
accuracy = compute_accuracy(y_test, y_pred_binary)
print(f"Model Accuracy: {accuracy * 100:.2f}%")


Model Accuracy: 75.46%


In [79]:
spikes_obj.spkMat.shape

(900750, 623)

In [80]:
900750/750

1201.0

In [81]:
np.save("F:/spikes.npy", spikes_obj.spkMat)